In [1]:
import pandas as pd
import numpy as np

In [2]:
lt = pd.read_csv("Data/Context/Lithuania.csv")

In [3]:
lt = lt.drop(["Unnamed: 0"], axis=1)

In [4]:
lt_2007_2013 = lt.loc[7:13]

In [5]:
lt_2007_2013

,TIME_PERIOD,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,...,WTDA,EDBS,DBP,DBD,SSB,DWCP,PMSI,PT,EC,ELFE
7,2007,0.69,0.66,0.10,1.01,0.89,48.0,0.962,0.964,49.9,...,3.98,NaN,7.0,26.0,81.367724,75.539488,50.000000,79.207745,77.271252,NaN
8,2008,0.68,0.56,0.11,1.04,0.85,46.0,0.962,0.964,55.2,...,4.03,NaN,7.0,26.0,81.503460,75.571061,50.000000,79.492010,77.271252,NaN
9,2009,0.72,0.69,0.23,0.95,0.90,49.0,0.967,0.964,59.8,...,3.93,NaN,7.0,26.0,82.186109,75.888508,50.000000,79.791757,77.271252,NaN
10,2010,0.78,0.74,0.37,0.96,0.92,50.0,0.970,0.970,62.1,...,3.85,73.141176,7.0,26.0,82.520151,75.934847,50.000000,80.789148,74.812235,NaN
11,2011,0.77,0.70,0.32,0.93,0.86,48.0,0.966,0.970,63.6,...,3.76,73.230806,6.0,22.0,84.636537,75.872411,50.000000,80.127684,74.812235,NaN
12,2012,0.84,0.83,0.39,1.12,0.93,54.0,0.966,0.970,67.2,...,3.78,74.330059,6.0,22.0,84.670001,77.991192,56.666667,81.101556,74.812235,NaN
13,2013,0.83,0.83,0.42,1.15,0.94,57.0,0.957,0.974,68.5,...,3.91,74.331592,7.0,19.5,84.317188,79.437628,56.666667,81.028515,72.899667,NaN


In [6]:
def missing_values(df):
    for feature in df.columns:
        missing_values = df[feature].isna().sum()
        percentage = missing_values/len(df) * 100
        print(f'{feature} - {missing_values} - {round(percentage, 1)} %')

In [7]:
missing_values(lt_2007_2013)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
VAA - 0 - 0.0 %
CPI - 0 - 0.0 %
FOP - 0 - 0.0 %
JI - 0 - 0.0 %
IUI - 0 - 0.0 %
CFI - 4 - 57.1 %
PR - 0 - 0.0 %
DI - 2 - 28.6 %
SOARS - 0 - 0.0 %
TOGP - 0 - 0.0 %
IPAB - 3 - 42.9 %
PTIP - 0 - 0.0 %
QOES - 0 - 0.0 %
AOLT - 0 - 0.0 %
FLTA - 0 - 0.0 %
EBOF - 0 - 0.0 %
FOWD - 0 - 0.0 %
QOMS - 0 - 0.0 %
CLER - 0 - 0.0 %
WTDA - 0 - 0.0 %
EDBS - 3 - 42.9 %
DBP - 0 - 0.0 %
DBD - 0 - 0.0 %
SSB - 0 - 0.0 %
DWCP - 0 - 0.0 %
PMSI - 0 - 0.0 %
PT - 0 - 0.0 %
EC - 0 - 0.0 %
ELFE - 7 - 100.0 %


In [8]:
mode = lt_2007_2013['DI'].mode()[0]
lt_2007_2013['DI'].fillna(value=mode, inplace=True)

/var/folders/41/w_0mz3gn2vn0w0dz8wywr8_r0000gn/T/ipykernel_34561/2997446236.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lt_2007_2013['DI'].fillna(value=mode, inplace=True)


In [9]:
lt_2007_2013 = lt_2007_2013.drop(["ELFE", "CFI", "IPAB", "EDBS"], axis=1)

In [10]:
def fill_with_mean(df):
    for feature in df.columns:
        mean = df[feature].mean()
        df[feature].fillna(value=mean, inplace=True)

In [11]:
continuous_data = ['DBP', "DBD", "EDBS", "SSB", "DWCP", "PT", "EC", "IUI", "PMSI"]

In [12]:
def handle_outliers(df, continuous_features):

    print("Original Shape:", df.shape)
    
    outlier_summary = {}
    
    for feature in continuous_features:
        if feature in df.columns:
            Q1 = df[feature].quantile(0.25)
            Q3 = df[feature].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outlier_mask = (df[feature] < lower_bound) | (df[feature] > upper_bound)
            num_outliers = outlier_mask.sum()
            
            if num_outliers > 0:
                non_outlier_mean = df.loc[~outlier_mask, feature].mean()
                
                df.loc[outlier_mask, feature] = non_outlier_mean
                
                print(f"{feature}: Found {num_outliers} outliers, replaced with mean = {non_outlier_mean:.2f}")
            else:
                print(f"{feature}: No outliers found")
    
    print("\nFinal Shape:", df.shape)
    print(f"Total features processed: {len(continuous_features)}")
    
    return df

In [13]:
lt_2007_2013_no_o = handle_outliers(lt_2007_2013, continuous_data)

Original Shape: (7, 30)
DBP: No outliers found
DBD: No outliers found
SSB: No outliers found
DWCP: Found 1 outliers, replaced with mean = 76.13
PT: No outliers found
EC: No outliers found
IUI: No outliers found
PMSI: No outliers found

Final Shape: (7, 30)
Total features processed: 9


In [14]:
lt_2007_2013_no_o

,TIME_PERIOD,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,...,QOMS,CLER,WTDA,DBP,DBD,SSB,DWCP,PMSI,PT,EC
7,2007,0.69,0.66,0.10,1.01,0.89,48.0,0.962,0.964,49.9,...,4.44,4.54,3.98,7.0,26.0,81.367724,75.539488,50.000000,79.207745,77.271252
8,2008,0.68,0.56,0.11,1.04,0.85,46.0,0.962,0.964,55.2,...,4.35,4.55,4.03,7.0,26.0,81.503460,75.571061,50.000000,79.492010,77.271252
9,2009,0.72,0.69,0.23,0.95,0.90,49.0,0.967,0.964,59.8,...,4.21,4.50,3.93,7.0,26.0,82.186109,75.888508,50.000000,79.791757,77.271252
10,2010,0.78,0.74,0.37,0.96,0.92,50.0,0.970,0.970,62.1,...,4.16,4.38,3.85,7.0,26.0,82.520151,75.934847,50.000000,80.789148,74.812235
11,2011,0.77,0.70,0.32,0.93,0.86,48.0,0.966,0.970,63.6,...,4.20,4.27,3.76,6.0,22.0,84.636537,75.872411,50.000000,80.127684,74.812235
12,2012,0.84,0.83,0.39,1.12,0.93,54.0,0.966,0.970,67.2,...,4.33,4.30,3.78,6.0,22.0,84.670001,77.991192,56.666667,81.101556,74.812235
13,2013,0.83,0.83,0.42,1.15,0.94,57.0,0.957,0.974,68.5,...,4.37,4.25,3.91,7.0,19.5,84.317188,76.132918,56.666667,81.028515,72.899667


In [15]:
lt_2007_2013_no_o.to_csv('Data/Context/Lithuania_2007_2013.csv')

In [16]:
lt_2007_2013_no_o = lt_2007_2013_no_o.drop(["TIME_PERIOD"], axis=1)

In [17]:
lt_2007_2013_l = lt_2007_2013_no_o.loc[13]

In [18]:
lt_2007_2013_l.to_csv('Data/Context/Lithuania_2007_2013_last.csv')

In [19]:
lt_2014_2020 = lt.loc[14:20]

In [20]:

missing_values(lt_2014_2020)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
VAA - 0 - 0.0 %
CPI - 0 - 0.0 %
FOP - 0 - 0.0 %
JI - 0 - 0.0 %
IUI - 0 - 0.0 %
CFI - 1 - 14.3 %
PR - 0 - 0.0 %
DI - 0 - 0.0 %
SOARS - 3 - 42.9 %
TOGP - 3 - 42.9 %
IPAB - 3 - 42.9 %
PTIP - 3 - 42.9 %
QOES - 3 - 42.9 %
AOLT - 3 - 42.9 %
FLTA - 3 - 42.9 %
EBOF - 3 - 42.9 %
FOWD - 3 - 42.9 %
QOMS - 3 - 42.9 %
CLER - 3 - 42.9 %
WTDA - 4 - 57.1 %
EDBS - 0 - 0.0 %
DBP - 0 - 0.0 %
DBD - 0 - 0.0 %
SSB - 0 - 0.0 %
DWCP - 0 - 0.0 %
PMSI - 0 - 0.0 %
PT - 0 - 0.0 %
EC - 0 - 0.0 %
ELFE - 1 - 14.3 %


In [21]:
lt_2014_2020 = lt_2014_2020.drop(["SOARS", "TOGP", "IPAB", "PTIP", "QOES", "AOLT", "FLTA", "EBOF",
                                 "FOWD", "QOMS", "CLER", "WTDA"], axis=1)

In [22]:
fill_with_mean(lt_2014_2020)

In [23]:
continuous_data = ['DBP', "DBD", "EDBS", "SSB", "DWCP", "PT", "EC", "IUI", "PMSI"]

In [24]:
lt_2014_2020_no_o = handle_outliers(lt_2014_2020, continuous_data)

Original Shape: (7, 22)
DBP: Found 1 outliers, replaced with mean = 4.17
DBD: Found 1 outliers, replaced with mean = 5.50
EDBS: No outliers found
SSB: Found 1 outliers, replaced with mean = 92.98
DWCP: No outliers found
PT: No outliers found
EC: Found 1 outliers, replaced with mean = 78.80
IUI: No outliers found
PMSI: No outliers found

Final Shape: (7, 22)
Total features processed: 9


In [25]:
lt_2014_2020_no_o.to_csv('Data/Context/Lithuania_2014_2020.csv')

In [26]:
lt_2014_2020_no_o = lt_2014_2020_no_o.drop(["TIME_PERIOD"], axis=1)

In [27]:
lt_2014_2020_no_o

,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,CFI,...,DI,EDBS,DBP,DBD,SSB,DWCP,PMSI,PT,EC,ELFE
14,0.94,0.98,0.55,1.19,0.96,58.0,0.943,0.974,72.1,41.000000,...,7.54,75.409557,4.166667,5.5,92.975890,79.489447,60.0,81.075004,78.802445,4.983333
15,0.98,1.15,0.58,1.26,0.97,61.0,0.953,0.974,71.4,42.300000,...,7.54,75.648605,5.000000,5.5,92.436369,79.333191,60.0,81.549466,78.802445,5.500000
16,1.00,1.03,0.68,1.12,1.00,59.0,0.948,0.974,74.4,41.800000,...,7.47,78.996605,4.000000,5.5,92.931332,79.315019,64.0,85.480409,78.802445,4.800000
17,0.96,0.93,0.52,1.15,0.99,59.0,0.893,0.972,77.6,41.200000,...,7.41,79.221080,4.000000,5.5,92.984744,79.323842,64.0,85.497006,78.802445,5.400000
18,0.92,1.03,0.47,1.09,0.90,59.0,0.910,0.967,79.7,41.166667,...,7.50,80.641847,4.000000,5.5,93.053500,84.759393,66.0,87.887522,78.802445,4.600000
19,0.99,1.01,0.67,1.15,0.99,60.0,0.910,0.967,81.6,41.500000,...,7.50,80.967168,4.000000,5.5,93.175686,84.854331,68.0,88.744373,78.802445,4.000000
20,0.95,1.01,0.78,1.08,1.00,60.0,0.905,0.967,83.1,39.200000,...,7.13,81.629700,4.000000,5.5,93.273709,84.870297,70.0,88.915657,78.802445,5.600000


In [28]:
lt_2014_2020_l = lt_2014_2020_no_o.loc[20]

In [29]:
lt_2014_2020_l

ROL      0.950000
GE       1.010000
COC      0.780000
RQ       1.080000
VAA      1.000000
CPI     60.000000
FOP      0.905000
JI       0.967000
IUI     83.100000
CFI     39.200000
PR      78.000000
DI       7.130000
EDBS    81.629700
DBP      4.000000
DBD      5.500000
SSB     93.273709
DWCP    84.870297
PMSI    70.000000
PT      88.915657
EC      78.802445
ELFE     5.600000
Name: 20, dtype: float64

In [30]:
lt_2014_2020_l.to_csv('Data/Context/Lithuania_2014_2020_last.csv')